# Introduction to Machine Learning

Machine Learning is the branch of artificial intelligence that enables computers to **learn from data** without being explicitly programmed for every scenario. Instead of a developer writing rules that say *if the email contains these words, mark it as spam*, a machine learning system studies thousands of labeled emails and discovers which patterns distinguish spam from legitimate mail. The rules are not written by humans — they are **learned** from examples.

This single idea — learning patterns from data to make predictions or decisions — powers much of the modern world. Recommendation systems suggest what you might watch next. Fraud detectors flag suspicious transactions. Medical models assist in diagnosis. Voice assistants recognize speech. Language models generate text. Self-driving cars perceive their environment. All of these rely on machine learning at their core.

This notebook introduces what machine learning is, how it differs from traditional programming and from broader AI, how a typical learning system works, the vocabulary you need to navigate the field, and the principles that guide when machine learning is the right tool — and when it is not.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. What Is Machine Learning?

Arthur Samuel, who built one of the first self-learning programs in 1959, defined machine learning as *"a field of study that gives computers the ability to learn without being explicitly programmed."* Tom Mitchell, in a more formal 1997 definition, stated that a computer program **learns** from experience $E$ with respect to some task $T$ and performance measure $P$, if its performance at $T$, as measured by $P$, improves with experience $E$.

Unpacking Mitchell's definition with a concrete example: a spam filter (**task** $T$) improves its accuracy at classifying emails (**performance** $P$) as it processes more emails and receives user feedback (**experience** $E$). The program was not updated by a developer rewriting rules — it improved through exposure to data.

At its core, machine learning is about building a **model** — a mathematical representation that captures patterns in data. The model takes an **input** (features describing a situation) and produces an **output** (a prediction, classification, or decision). During **training**, the model adjusts its internal parameters to minimize errors on known examples. During **inference** (also called prediction), the trained model applies what it learned to new, unseen inputs.

$$\text{Training:} \quad \text{Data} + \text{Labels} \rightarrow \text{Learned Model}$$

$$\text{Prediction:} \quad \text{New Input} + \text{Learned Model} \rightarrow \text{Output}$$

This paradigm shift — from explicit rules to learned patterns — is what makes machine learning powerful for problems where the rules are too complex, too numerous, or too subtle for humans to write by hand.

---

## 2. Machine Learning in the Broader AI Landscape

Machine learning does not exist in isolation. It sits within the larger field of artificial intelligence and connects to several related disciplines.

```
┌─────────────────────────────────────────────────────────┐
│              ARTIFICIAL INTELLIGENCE (AI)               │
│  ┌───────────────────────────────────────────────────┐  │
│  │           MACHINE LEARNING (ML)                   │  │
│  │  ┌─────────────────────────────────────────────┐  │  │
│  │  │         DEEP LEARNING (DL)                  │  │  │
│  │  └─────────────────────────────────────────────┘  │  │
│  └───────────────────────────────────────────────────┘  │
│  Also includes: search, logic, expert systems, planning │
└─────────────────────────────────────────────────────────┘
```

**Artificial Intelligence** is the broad goal of creating systems that perform tasks requiring intelligence. **Machine Learning** is a subset of AI focused on systems that improve through experience with data. **Deep Learning** is a subset of machine learning that uses multi-layer neural networks. Not all AI uses machine learning — a GPS pathfinding algorithm uses classical search, not learning. Not all machine learning is deep learning — a decision tree or linear regression model can be highly effective without neural networks.

In industry and media, the terms are often used interchangeably, but the distinctions matter. When someone says "we use AI," they usually mean machine learning. When they say "deep learning," they mean neural networks with many layers. Precision in terminology helps you understand what technology is actually being deployed and what skills are required to build or maintain it.

---

## 3. Traditional Programming vs Machine Learning

The fundamental difference between traditional programming and machine learning is **where the knowledge comes from**.

### 3.1 Traditional Programming

In traditional programming, a human analyst understands the problem, identifies the logic, and writes explicit instructions. The computer executes those instructions on input data to produce output.

```
Rules (written by human) + Data → Output
```

Example: A payroll system calculates tax using formulas defined by tax law. Every case is handled by rules the programmer encoded. If a new tax bracket is introduced, a developer must update the code.

### 3.2 Machine Learning

In machine learning, the human provides examples of inputs and desired outputs (or just inputs, in unsupervised settings). The algorithm discovers the mapping on its own.

```
Data + Desired Outputs → Learned Model → Predictions on New Data
```

Example: A house price predictor learns from thousands of past sales — each with features like area, bedrooms, and location, and the actual sale price. No one writes a formula for price; the model learns the relationship from data. When market conditions shift, retraining on recent data can adapt the model without rewriting logic.

### 3.3 When Each Approach Applies

Traditional programming excels when rules are **known, stable, and finite**. Tax calculation, password validation, and file format parsing are rule-based problems. Machine learning excels when the pattern is **complex, implicit, or evolving** — recognizing faces, predicting customer churn, translating languages, or detecting anomalies in network traffic. Many production systems combine both: hard rules for known constraints and learned models for pattern recognition.

In [ ]:
# Rule-based vs learned: predicting pass/fail from study hours

study_hours = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10])
passed = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])  # 1 = pass

# Rule-based: human decides threshold
def rule_based_predict(hours, threshold=5):
    return 1 if hours >= threshold else 0

# Learned: find best threshold from data (simplest possible "model")
best_threshold, best_accuracy = 5, 0
for t in range(1, 10):
    preds = (study_hours >= t).astype(int)
    acc = (preds == passed).mean()
    if acc > best_accuracy:
        best_accuracy, best_threshold = acc, t

print("Rule-based (human picks threshold=5):")
for h in [3, 5, 8]:
    print(f"  {h} hours → {'Pass' if rule_based_predict(h) else 'Fail'}")

print(f"\nLearned from data (best threshold={best_threshold}, accuracy={best_accuracy:.0%}):")
for h in [3, 5, 8]:
    print(f"  {h} hours → {'Pass' if (h >= best_threshold) else 'Fail'}")

---

## 4. How Machine Learning Works: The Core Loop

Every machine learning project, regardless of algorithm or domain, follows a common structure. Understanding this loop is essential before diving into specific techniques.

### 4.1 Gather Data

Machine learning requires data — often large amounts of it. Data can be structured (tables with rows and columns), unstructured (text, images, audio), or semi-structured (JSON, logs). The quality, quantity, and relevance of data largely determine the success of a project. Poor data produces poor models regardless of algorithm sophistication.

### 4.2 Prepare Data

Raw data is rarely ready for training. Preparation includes cleaning (handling missing values, correcting errors), transforming (scaling features, encoding categories), splitting into training and test sets, and sometimes augmenting (creating additional examples). Data preparation often consumes more time than model training.

### 4.3 Choose a Model

A **model** is an algorithm with adjustable **parameters**. Linear regression, decision trees, support vector machines, and neural networks are all models. The choice depends on the problem type (regression, classification, clustering), data size, interpretability requirements, and computational constraints.

### 4.4 Train the Model

Training feeds data to the model and adjusts parameters to minimize a **loss function** — a number measuring how wrong the predictions are. For regression, loss might be mean squared error. For classification, cross-entropy loss. The training algorithm iteratively improves parameters until loss stops decreasing meaningfully.

### 4.5 Evaluate the Model

Evaluation measures performance on data the model has **not seen** during training. This estimates how well the model will generalize to real-world inputs. Metrics include accuracy, precision, recall, F1 score, mean squared error, and R², depending on the task.

### 4.6 Deploy and Monitor

A trained model is integrated into a product or pipeline where it makes predictions on live data. Monitoring tracks performance over time — models can degrade as data distributions change (**concept drift**), requiring periodic retraining.

In [ ]:
# Minimal ML loop: predict house prices from area (synthetic data)

np.random.seed(42)
area = np.random.uniform(800, 2500, 80)          # sq ft
price = 50 + 0.12 * area + np.random.normal(0, 20, 80)  # price in thousands

# Train: fit a linear model (price = w * area + b)
A = np.column_stack([area, np.ones(len(area))])
w, b = np.linalg.lstsq(A, price, rcond=None)[0]

# Predict on new houses
new_areas = np.array([1000, 1500, 2000])
predictions = w * new_areas + b

print(f"Learned model: price = {w:.4f} × area + {b:.2f}")
print("\nPredictions for new houses:")
for a, p in zip(new_areas, predictions):
    print(f"  {a:.0f} sq ft → ${p:.1f}k")

plt.figure(figsize=(8, 4))
plt.scatter(area, price, alpha=0.6, label="Training data")
x_line = np.linspace(area.min(), area.max(), 100)
plt.plot(x_line, w * x_line + b, "r-", linewidth=2, label="Learned model")
plt.xlabel("Area (sq ft)")
plt.ylabel("Price ($ thousands)")
plt.title("Machine Learning: Learn a Pattern from Data")
plt.legend()
plt.show()

---

## 5. Types of Machine Learning

Machine learning algorithms are grouped by the type of signal available during training. Each type addresses different kinds of problems.

### 5.1 Supervised Learning

The algorithm learns from **labeled** examples — input-output pairs. For each training example, the correct answer is known. Supervised learning splits into:

- **Regression:** Predict a continuous value (house price, temperature, stock price).
- **Classification:** Predict a category (spam or not spam, disease or healthy, cat or dog).

Supervised learning is the most common paradigm in industry because labeled data directly defines what "correct" means.

### 5.2 Unsupervised Learning

The algorithm learns from **unlabeled** data — inputs without correct answers. The system discovers hidden structure:

- **Clustering:** Group similar data points (customer segments, document topics).
- **Dimensionality reduction:** Compress data while preserving important information (visualization, noise removal).
- **Anomaly detection:** Identify unusual patterns (fraud, equipment failure).

### 5.3 Reinforcement Learning

An **agent** learns by interacting with an **environment**, receiving **rewards** or **penalties** for its actions. There are no labeled examples — only feedback on whether actions were good or bad. Applications include game playing, robotics, recommendation systems, and training language models with human feedback (RLHF).

### 5.4 Other Paradigms

**Semi-supervised learning** uses a small amount of labeled data combined with a large amount of unlabeled data — common when labeling is expensive. **Self-supervised learning** creates labels from the data itself (predicting the next word in a sentence) and powers large language model pre-training.

---

## 6. Essential Terminology

Machine learning has a precise vocabulary. These terms appear in every algorithm, paper, and tool.

**Feature** — An individual measurable property of a data point. For a house: square footage, number of bedrooms, age. Features are the inputs to a model.

**Label / Target** — The correct output for a supervised learning example. The price of a house, the category of an email, the diagnosis for a patient.

**Sample / Instance / Example** — A single row of data — one observation with its features (and optionally its label).

**Dataset** — The complete collection of samples used for training and evaluation.

**Training set** — Data used to fit the model parameters.

**Test set** — Data held out during training, used only for final evaluation. The model never sees test labels during training.

**Validation set** — Data used to tune hyperparameters and compare models during development.

**Model** — The learned representation mapping features to predictions.

**Parameters** — Values the model learns during training (weights in a neural network, coefficients in regression).

**Hyperparameters** — Settings chosen before training (learning rate, tree depth, number of neighbors in KNN). Not learned from data.

**Loss function** — Measures prediction error during training. The model minimizes this.

**Overfitting** — The model memorizes training data including noise and fails on new data.

**Underfitting** — The model is too simple to capture the underlying pattern.

**Generalization** — The model's ability to perform well on unseen data. The central goal of machine learning.

In [ ]:
# Terminology in practice: a tiny labeled dataset

data = {
    "area_sqft":   [1200, 1800, 950, 2100, 1500],
    "bedrooms":    [2, 3, 2, 4, 3],
    "age_years":   [15, 8, 22, 5, 12],
    "price_k":     [280, 410, 220, 520, 350],  # label / target
}

import pandas as pd
df = pd.DataFrame(data)

features = ["area_sqft", "bedrooms", "age_years"]
target = "price_k"

print("Dataset (5 samples):")
print(df.to_string(index=False))
print(f"\nFeatures (inputs): {features}")
print(f"Target (label):    {target}")
print(f"Number of samples: {len(df)}")

---

## 7. Regression vs Classification

The two main supervised learning tasks differ in the nature of the output.

**Regression** predicts a **continuous numerical value**. The output can be any number on a scale: $184,500 for a house price, 72.3°F for temperature, 0.87 for a probability. Algorithms include linear regression, polynomial regression, ridge regression, and regression trees.

**Classification** predicts a **discrete category or class**. The output is one of a fixed set of labels: spam or ham, cat or dog or bird, approved or denied. Binary classification has two classes; multi-class classification has three or more. Algorithms include logistic regression, K-nearest neighbors, decision trees, support vector machines, and naive Bayes.

Some problems can be framed either way. Predicting whether a loan will default is classification (yes/no). Predicting the probability of default is regression (a number between 0 and 1). The framing affects which algorithms and metrics you use.

In [ ]:
# Regression vs classification on the same features

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import StandardScaler

X = df[["area_sqft", "bedrooms", "age_years"]].values

# Regression: predict exact price
y_reg = df["price_k"].values
reg_model = LinearRegression().fit(X, y_reg)
reg_pred = reg_model.predict([[1600, 3, 10]])[0]

# Classification: predict expensive (1) vs affordable (0), threshold $350k
y_cls = (df["price_k"] >= 350).astype(int).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
cls_model = LogisticRegression().fit(X_scaled, y_cls)
cls_pred = cls_model.predict([[1600, 3, 10]])[0]
cls_prob = cls_model.predict_proba(scaler.transform([[1600, 3, 10]]))[0, 1]

print("For a house: 1600 sqft, 3 bed, 10 years old")
print(f"  Regression prediction:  ${reg_pred:.1f}k")
print(f"  Classification:        {'Expensive' if cls_pred == 1 else 'Affordable'} (P={cls_prob:.2f})")

---

## 8. Real-World Applications of Machine Learning

Machine learning is embedded across industries. Understanding where it is applied clarifies why the field matters and what kinds of problems you will encounter.

**Healthcare** — Disease diagnosis from medical images, drug discovery, patient risk stratification, personalized treatment recommendations.

**Finance** — Fraud detection, credit scoring, algorithmic trading, insurance underwriting, customer churn prediction.

**Retail and e-commerce** — Product recommendations, demand forecasting, dynamic pricing, inventory optimization, customer segmentation.

**Technology** — Search ranking, spam filtering, speech recognition, machine translation, content moderation, ad targeting.

**Manufacturing** — Predictive maintenance, quality inspection, supply chain optimization, defect detection.

**Transportation** — Route optimization, demand prediction for ride-sharing, autonomous vehicle perception.

**Agriculture** — Crop disease detection, yield prediction, precision irrigation.

The common thread is that each application involves patterns in data that are too complex for manual rules but learnable from examples.

---

## 9. When to Use Machine Learning

Machine learning is powerful but not universal. Use it when these conditions hold:

**The problem involves pattern recognition or prediction.** If you need to classify, rank, forecast, or detect anomalies in complex data, ML is appropriate.

**Sufficient data exists or can be collected.** Most algorithms need hundreds to millions of examples. Without data, there is nothing to learn from.

**The patterns may change over time.** ML models can be retrained on fresh data. Rule-based systems require manual updates for every change.

**Approximate answers are acceptable.** ML produces probabilistic, best-effort predictions — not guaranteed correct answers. High-stakes decisions often require human oversight.

**Do not use machine learning when:**

- Simple rules fully describe the problem (use traditional code).
- No data is available and cannot be obtained ethically.
- Perfect accuracy and full explainability are legally required with no human review.
- The cost of errors is catastrophic and the model cannot be validated.
- The problem is entirely deterministic with known logic.

---

## 10. Common Misconceptions

**"Machine learning is the same as AI."** Machine learning is a subset of AI. AI also includes symbolic reasoning, search, planning, and expert systems that do not learn from data.

**"More data always means a better model."** Data quality matters more than quantity. Noisy, biased, or irrelevant data produces poor models. Ten thousand well-labeled representative examples may outperform ten million poor ones.

**"ML models are objective."** Models learn from historical data, which encodes historical biases. A hiring model trained on past decisions may perpetuate past discrimination unless carefully audited.

**"Once trained, a model works forever."** Real-world data changes. A model trained on pre-pandemic shopping behavior may fail post-pandemic. Models need monitoring and periodic retraining.

**"ML replaces human expertise."** ML augments human decision-making. A medical AI assists radiologists; it does not replace them. The most effective systems combine human judgment with machine predictions.

**"You need a PhD to do machine learning."** Modern libraries (scikit-learn, PyTorch, TensorFlow) make it accessible to developers with solid fundamentals in math and programming. Deep expertise helps for research and novel problems, but many practical applications are within reach of dedicated practitioners.

---

## 11. The Machine Learning Ecosystem

Practitioners work with a rich ecosystem of tools and libraries.

**NumPy** — Numerical computing foundation. Arrays, linear algebra, random number generation.

**Pandas** — Data manipulation and analysis. Tables, filtering, grouping, missing value handling.

**Matplotlib / Seaborn** — Data visualization. Histograms, scatter plots, heatmaps.

**Scikit-learn** — Classical machine learning algorithms. Regression, classification, clustering, preprocessing, model evaluation — the standard starting point.

**PyTorch / TensorFlow** — Deep learning frameworks. Neural networks, GPUs, production deployment.

**Jupyter Notebooks** — Interactive environment for exploration, visualization, and documentation — like this one.

A typical workflow uses Pandas for data exploration, scikit-learn for classical ML, and PyTorch or TensorFlow when deep learning is needed.

In [ ]:
# End-to-end with scikit-learn: train, evaluate, predict

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Larger synthetic dataset
np.random.seed(42)
n = 200
X_full = np.column_stack([
    np.random.uniform(800, 2500, n),
    np.random.randint(1, 5, n),
    np.random.randint(0, 30, n),
])
y_full = 50 + 0.1 * X_full[:, 0] - 2 * X_full[:, 2] + np.random.normal(0, 25, n)

X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Scikit-learn ML pipeline:")
print(f"  Training samples:   {len(X_train)}")
print(f"  Test samples:       {len(X_test)}")
print(f"  Test MSE:           {mean_squared_error(y_test, y_pred):.2f}")
print(f"  Test R²:            {r2_score(y_test, y_pred):.4f}")
print(f"\n  Coefficients (area, bedrooms, age): {model.coef_.round(4)}")
print(f"  Intercept: {model.intercept_:.2f}")

---

## 12. Overfitting and Generalization Preview

The central challenge in machine learning is **generalization** — performing well on data the model has never seen. Two failure modes dominate:

**Underfitting** occurs when the model is too simple to capture the pattern. A straight line fit to clearly curved data will underfit. Training error and test error are both high.

**Overfitting** occurs when the model is too complex relative to the amount of data. It memorizes training examples, including noise, and fails on new data. Training error is low but test error is high.

The goal is the sweet spot: a model complex enough to capture real patterns but not so complex that it memorizes noise. Techniques for managing this tradeoff — regularization, cross-validation, more data, simpler models — are covered in later notebooks.

In [ ]:
# Visualizing underfitting vs overfitting

np.random.seed(42)
x = np.linspace(0, 1, 20)
y = np.sin(2 * np.pi * x) + np.random.normal(0, 0.15, 20)
x_plot = np.linspace(0, 1, 200)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
degrees = [1, 4, 15]
titles = ["Underfit (degree=1)", "Good fit (degree=4)", "Overfit (degree=15)"]

for ax, deg, title in zip(axes, degrees, titles):
    coeffs = np.polyfit(x, y, deg)
    ax.scatter(x, y, color="steelblue", zorder=3)
    ax.plot(x_plot, np.polyval(coeffs, x_plot), "r-", linewidth=2)
    ax.set_title(title)
    ax.set_ylim(-2, 2)

plt.suptitle("Bias-Variance Tradeoff", y=1.02)
plt.tight_layout()
plt.show()

---

## 13. Summary

Machine learning enables computers to learn patterns from data rather than following hand-written rules. It is a subset of artificial intelligence, distinct from but related to deep learning and classical AI approaches. The core loop — gather data, prepare it, choose a model, train, evaluate, deploy — underlies every ML project.

Supervised learning uses labeled examples for regression and classification. Unsupervised learning finds structure without labels. Reinforcement learning trains agents through rewards and penalties. The vocabulary of features, labels, training sets, loss functions, overfitting, and generalization appears throughout the field.

Machine learning excels at pattern recognition and prediction when sufficient quality data exists and approximate answers are acceptable. It does not replace traditional programming for rule-based problems or human judgment for high-stakes decisions. Tools like NumPy, Pandas, and scikit-learn make the practice accessible; understanding the concepts makes the practice effective.

The notebooks that follow build on this foundation — exploring each learning paradigm, preprocessing technique, algorithm family, and evaluation method in the depth needed to build real machine learning systems.